# DuckDB Python Client

In [1]:
import duckdb
import pandas as pd
from faker import Faker

The DuckDB Python client exposes a "limited" set of functions to (not exhaustive):

- Read / write datasets.
- Run SQL commands.
- Scan the response.

Most of the power lies in the SQL language.

In [2]:
col_size = 7

functions = [f for f in dir(duckdb) if not f.startswith('_') and f.islower()]

for i in range(0, len(functions), col_size):
    print(', '.join(functions[i: i + col_size]))

abort, aggregate, alias, apilevel, append, appender, array_type
arrow, begin, build_info, checkpoint, close, coalesce, col
commit, complete, connect, count_all, create_function, cursor, decimal_type
default_connection, dense_rank, description, df, disable_profiling, distinct, dtype
duplicate, enable_profiling, enum_type, execute, executemany, extract_statements, fetch_df
fetch_df_chunk, fetchall, fetchdf, fetchmany, fetchnumpy, fetchone, filesystem_is_registered
filter, first_value, fn, from_arrow, from_csv_auto, from_df, from_parquet
from_query, get_profiling_information, get_table_names, install_extension, interrupt, lag, last_value
lead, limit, list_filesystems, list_type, lit, load_extension, map_type
ntile, order, paramstyle, project, query, query_appender, query_df
query_progress, rank, read_csv, read_json, read_parquet, register, register_filesystem
remove_function, rollback, row_number, row_type, rowcount, set_default_connection, sql
sql_expr, sqltype, sqltypes, star, string_ty

## Objects Lifetime

When using DuckDB through `duckdb.sql()`, it operates on an **in-memory** database.

In [3]:
duckdb.sql('CREATE OR REPLACE TABLE t AS SELECT 42 AS value')

CountBIGINT
1


The objects created can be **reused** later (i.e. persisted in the in-memory database).

In [4]:
duckdb.sql('SELECT value, value ** 2 AS square FROM t')

valueINTEGER,squareDOUBLE
42,1764.0


💡 Work with an explicit connection ([`duckdb.connect()`](https://duckdb.org/docs/current/clients/python/reference/#duckdb.connect)) or a [named in-memory connection](https://duckdb.org/docs/current/clients/python/dbapi#named-in-memory-connections) to **isolate** memory sessions.

## DataFrames

### DataFrame Access

DuckDB can **directly** query [Pandas](https://pandas.pydata.org/) or [Polars](https://pola.rs/) DataFrames and [Arrow](https://arrow.apache.org/docs/cpp/tables.html) tables.  
Thanks to [Apache Arrow](https://arrow.apache.org/) (fast data interchange and in-memory analytics).

In [5]:
df = pd.DataFrame(
    {
        'name': ['lorem', 'ipsum', 'dolor'],
        'value': [1, 2, 3],
    }
)

df

,name,value
0,lorem,1
1,ipsum,2
2,dolor,3


In [6]:
# Query the dataframe created in the Python code using the DuckDB SQL language.
duckdb.sql('FROM df')

nameVARCHAR,valueBIGINT
lorem,1
ipsum,2
dolor,3


In [7]:
duckdb.sql('SUMMARIZE df')

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
name,VARCHAR,dolor,lorem,3,NULL,NULL,NULL,NULL,NULL,3,0.00
value,BIGINT,1,3,3,2.0,1.0,1,2,3,3,0.00


Query with [named parameters](https://duckdb.org/docs/current/clients/python/dbapi#named-parameters).

In [8]:
duckdb.sql(
    'FROM df WHERE name = $name',
    params={'name': 'ipsum'},
)

nameVARCHAR,valueBIGINT
ipsum,2


### DataFrame Creation

DataFrames are **read-only**!

💡 Tables created in DuckDB can be returned as DataFrame.

In [9]:
df2 = duckdb.sql("""
    CREATE OR REPLACE TABLE updated AS FROM df;
    INSERT INTO updated VALUES ('sit', 4);
    FROM updated;
""").df()

df2

,name,value
0,lorem,1
1,ipsum,2
2,dolor,3
3,sit,4


### Table Update

Using the [Python appender](https://duckdb.org/2026/10/26/gems-of-duckdb-200#python-appender).

In [10]:
duckdb.sql("""
    CREATE OR REPLACE TABLE updated AS FROM df;

    FROM updated;
""").df()

,name,value
0,lorem,1
1,ipsum,2
2,dolor,3


In [11]:
with duckdb.appender('updated') as appender:
    appender.append_rows([
        ('sit', 4),
        ('amet', 5),
    ])

duckdb.sql('FROM updated')

nameVARCHAR,valueBIGINT
lorem,1
ipsum,2
dolor,3
sit,4
amet,5


## User Defined Functions

**User Defined Functions** can be implemented in [Python](https://duckdb.org/docs/current/clients/python/function).

In [12]:
def get_random_name() -> str:
    fake = Faker()
    return fake.name()

In [13]:
get_random_name()

'Pamela Miles'

By default, functions are expected to produce the same result for the same input.

💡 If the result of a function is impacted by any type of randomness, [`side_effects`](https://duckdb.org/docs/current/clients/python/function#side-effects) must be set to `True`.

In [14]:
try:
    duckdb.remove_function('random_name')
except duckdb.InvalidInputException:
    ...

duckdb.create_function(
    'random_name',
    get_random_name,
    side_effects=True,
)

In [15]:
duckdb.sql("""
    SELECT random_name() AS name
    FROM unnest(range(10))
""")

nameVARCHAR
Jeremiah Dunn
Richard Neal
Donna Leonard
Kenneth Smith
Elizabeth Williams
Justin Chavez
John Vega
Rebecca Anderson
Mary Mosley
Ana Price
